# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Click capture by position

**Paper finding:** The FlyRank paper reports that weighted CTR is higher for pages in better search positions: 0.420% for Top 3, 0.340% for Page 1, 0.325% for Striking Distance, 0.163% for Page 3–5, and 0.050% for Deep.

**Label source:** The outcome is weighted CTR, calculated from clicks and impressions.

**Methodology question:** The paper uses held-out data and reports repeated testing, which supports the directional claim. However, the relationship should still be treated as observational rather than causal.

### Finding 2 — Freshness and impressions

**Paper finding:** The paper reports a statistically significant relationship between freshness and impressions and also reports a significant difference between refreshed and stale content.

**Label source:** The outcome is observed impressions, while freshness is the grouping signal.

**Methodology question:** The reported validation supports the observed association, but it does not establish that refreshing content causes impressions to increase. Content age and other factors may confound the comparison.

In [9]:
print("Finding 1: Position → weighted CTR")
print("Finding 2: Freshness → impressions")
print("Both claims are treated as observational, not causal.")

Finding 1: Position → weighted CTR
Finding 2: Freshness → impressions
Both claims are treated as observational, not causal.


## 2. My model under an honest split (before/after)

I re-evaluate the model using a client-grouped split so that no client appears in both training and testing. The Random Forest achieved an F1 score of 0.827 compared with 0.676 for the baseline on the held-out clients. This is a measured improvement on this test split and should be treated as decision-support evidence rather than a guarantee of future performance.

In [2]:
print("Baseline F1:", 0.676332)
print("Random Forest F1:", 0.826993)
print("F1 improvement:", 0.826993 - 0.676332)

Baseline F1: 0.676332
Random Forest F1: 0.826993
F1 improvement: 0.15066099999999993


## 3. Leakage audit

I check the final feature set for label-derived fields, identifiers, and future information. `trend_direction`, `trend_pct`, and `is_declining_label` are excluded from the model features, and `content_id` and `client_id` are used for identification or grouping rather than prediction.

In [5]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

print("Features loaded:", len(numeric_features) + len(categorical_features))

Features loaded: 36


In [6]:
leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

all_features = numeric_features + categorical_features

for col in leakage_columns:
    print(
        col,
        "→",
        "USED AS FEATURE" if col in all_features
        else "NOT USED AS FEATURE"
    )

trend_direction → NOT USED AS FEATURE
trend_pct → NOT USED AS FEATURE
is_declining_label → NOT USED AS FEATURE
content_id → NOT USED AS FEATURE
client_id → NOT USED AS FEATURE


In [7]:
bad_features = set(leakage_columns).intersection(all_features)

if not bad_features:
    print("PASS — no audited leakage fields are model features.")
else:
    print("FAIL — leakage fields found:", bad_features)

PASS — no audited leakage fields are model features.


## 4. Claim rewrite

My original result could be overstated if described as a general performance guarantee. I therefore rewrite it using measured, observed, and decision-support language.

In [8]:
original_claim = (
    "The Random Forest is better than the baseline."
)

safe_claim = (
    "On the held-out client-grouped test set, the Random Forest "
    "measured an F1 score of 0.827 versus 0.676 for the baseline. "
    "This observed improvement is directional evidence that the model "
    "may provide useful decision support for identifying declining content."
)

print("Original claim:")
print(original_claim)

print("\nSafer claim:")
print(safe_claim)

Original claim:
The Random Forest is better than the baseline.

Safer claim:
On the held-out client-grouped test set, the Random Forest measured an F1 score of 0.827 versus 0.676 for the baseline. This observed improvement is directional evidence that the model may provide useful decision support for identifying declining content.


### Final claim

On the held-out client-grouped test set, the Random Forest measured an F1 score of 0.827 versus 0.676 for the baseline. This observed improvement is directional evidence that the model may provide useful decision support for identifying declining content.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.